# Water Body Detection: download data, train every model, compare results

Run top to bottom on Google Colab (about 30 to 40 minutes).

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. You need a free **Earth Engine** project. Go to https://code.earthengine.google.com/register, choose **Unpaid usage → Academia & research**, and create a Cloud project. Copy its **project ID** (for example `ee-gyanasai`).

What it does:
1. Downloads cloud-free Sentinel-2 and Landsat-8 scenes for 8 areas (lakes, rivers, lagoons, wetlands), with JRC water labels for the same month
2. Cuts them into tiles with a spatial train / validation / test split
3. Trains two index baselines, five classical models and a U-Net CNN on each sensor
4. Prints a comparison table and chart with your real results

## 1. Get the code

In [ ]:
!git clone -q https://github.com/gyanasaiadusumilli/water-body-detection.git
%cd water-body-detection
!pip install -q rasterio tabulate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none. Switch the runtime to T4 GPU first.")

## 2. Sign in to Earth Engine
Paste your Earth Engine project ID below, then run the cell and follow the sign-in prompt.

In [ ]:
PROJECT = "your-earth-engine-project-id"  # <- replace this

import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
print("Earth Engine ready")

## 3. Download the scenes (about 5 minutes)

In [ ]:
!python gee/fetch_scenes.py --project "{PROJECT}" 

## 4. Tile and split

In [ ]:
!python -m src.prepare_data --sensor sentinel2
!python -m src.prepare_data --sensor landsat8

## 5. Baselines and classical models

In [ ]:
!python -m src.train_ml --sensor sentinel2
!python -m src.train_ml --sensor landsat8

## 6. U-Net CNN (about 10 minutes per sensor on a T4)

In [ ]:
!python -m src.train_cnn --sensor sentinel2 --epochs 40
!python -m src.train_cnn --sensor landsat8 --epochs 40

## 7. Compare everything
Copy the table and send it to Claude so the README and portfolio show your real numbers.

In [ ]:
!python -m src.compare
print()
print("=== WATER BODY DETECTION RESULTS ===")
print(open("results/comparison.md").read())
from IPython.display import Image, display
display(Image("results/comparison_iou.png", width=640))

## 8. Map water across a full scene

In [ ]:
from pathlib import Path
scene = sorted(Path("data/raw/sentinel2").glob("hyderabad*.tif"))[0]
!python -m src.predict_scene "{scene}" --model models/sentinel2/cnn_unet.pt
preview = sorted(Path("predictions").glob("*_preview.png"))[-1]
display(Image(str(preview), width=900))

## 9. Download your results
Send the zip to Claude to add the results and figures to GitHub.

In [ ]:
!zip -q -r water_results.zip results predictions/*_preview.png
from google.colab import files
files.download("water_results.zip")